# CALCUALTE BIOSAMPLE COLLECTION AGE 

The script calculates both the biosample collection age for all individuals in the AoU. The duration in days is then converted to years by dividing by 365.25. 

In [75]:
# Import the packages
import os
import pandas as pd
from datetime import datetime
from google.cloud import bigquery
from functools import reduce

In [76]:
# Retrieve the value of the environment variable 'WORKSPACE_BUCKET' and assign it to the variable 'my_bucket'.
my_bucket = os.getenv("WORKSPACE_BUCKET")
# Display the value of 'my_bucket'
my_bucket


'gs://fc-secure-bf507049-90b6-43f8-86ea-c3ad9e823ffe'

In [77]:
!gsutil -u $GOOGLE_PROJECT cp gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/aux/qc/genomic_metrics.tsv .

Copying gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/aux/qc/genomic_metrics.tsv...
- [1 files][ 42.3 MiB/ 42.3 MiB]                                                
Operation completed over 1 objects/42.3 MiB.                                     


In [78]:
genomic_info = pd.read_csv('genomic_metrics.tsv', sep='\t')

In [79]:
import pandas
import os

# This query represents dataset "data_of_birth_ehr_cohort" for domain "person" and was generated for All of Us Controlled Tier Dataset v8
dataset_73356962_person_sql = """
    SELECT
        person.person_id,
        person.birth_datetime as date_of_birth 
    FROM
        `""" + os.environ["WORKSPACE_CDR"] + """.person` person   
    WHERE
        person.PERSON_ID IN (SELECT
            distinct person_id  
        FROM
            `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` cb_search_person  
        WHERE
            cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
            WHERE
                has_ehr_data = 1 ) )"""

dataset_73356962_person_df = pandas.read_gbq(
    dataset_73356962_person_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")

dataset_73356962_person_df.head(5)

Downloading:   0%|          | 0/393601 [00:00<?, ?rows/s]

,person_id,date_of_birth
0,1581351,2003-06-15 00:00:00+00:00
1,4521223,2003-06-15 00:00:00+00:00
2,1691808,1931-06-15 00:00:00+00:00
3,6152588,2001-06-15 00:00:00+00:00
4,2922960,1937-06-15 00:00:00+00:00


In [80]:
genomic_info.rename(columns={'research_id':'person_id'}, inplace=True)

In [81]:
genomic_info = (
genomic_info.merge(dataset_73356962_person_df, 
                         on ="person_id", 
                         how ="inner")
)

In [82]:
# Convert both columns to datetime and remove any timezone information (make them naive)
genomic_info["biosample_collection_date"] = (
    pd.to_datetime(genomic_info["biosample_collection_date"]).dt.tz_localize(None)
)

genomic_info["date_of_birth"] = (
    pd.to_datetime(genomic_info["date_of_birth"]).dt.tz_localize(None)
)


In [83]:
genomic_info["age"] = (
(genomic_info["biosample_collection_date"] - 
 genomic_info["date_of_birth"]).dt.days / 365.25
)

In [84]:
genomic_info.to_csv(f"{my_bucket}/data/covariates/biosample_collection_age.tsv", 
                    sep ='\t', 
                    index=False)